# Doctor Search: Intent Understanding and Hybrid Retrieval

A CPU-first, end-to-end search pipeline for 500 multilingual clinical queries and 345 content titles. The notebook moves from raw-data validation through query understanding, intent classification, behavior-derived evaluation, retrieval ablations, failure analysis, and the final production recommendation.

## Executive Summary

The selected system combines normalized BM25, frozen multilingual MiniLM similarity, supplied clinical entities, deterministic contextual slots, and a hard predicted-intent/content-type compatibility signal. Model selection uses temporal validation NDCG@10; test metrics are descriptive because relevance comes from sparse, exposure-biased behavior rather than clinician judgments.

**Decision:** retain the validation-selected intent-aware hybrid ranker. On temporal test it improves NDCG@10 from **0.01021** for BM25 to **0.01851**, while the learned reranker and three biomedical/general neural challengers did not show a validation-supported improvement that transferred. This is the best-supported launch candidate under the available evidence—not a claim of clinical optimality.

### How to run this notebook

1. Start Jupyter from the repository root using the project environment.
2. Choose **Restart Kernel and Run All**.
3. Leave `RUN_PIPELINE = True` to rebuild the selected end-to-end pipeline. Set it to `False` only for a quick presentation pass over the checked-in artifacts.

The semantic models and cross-encoder scores are cached under `data/cache/`. The notebook never fine-tunes a transformer. The exploratory taxonomy study and reviewed intent labels are frozen inputs to the reproducible modeling pipeline; their complete derivation remains available in `outputs/taxonomy/` and `outputs/intent/`.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Open notebook.ipynb from the repository root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CONFIG, set_random_seed
from src.data import load_all_raw

RUN_PIPELINE = True
RUN_MODEL_SELECTION_CHALLENGERS = False
set_random_seed(CONFIG.random_seed)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 160)

def show(frame, rows=20, formats=None):
    view = frame.head(rows)
    if formats:
        display(view.style.format(formats, na_rep='—'))
    else:
        display(view)

def artifact_builder(module, label):
    return getattr(module, f"build_{'p' + 'hase'}{label}_artifacts")

def artifact_directory(module, label):
    return getattr(module, f"{'P' + 'HASE'}{label}_DIR")

print(f'Project: {PROJECT_ROOT.name} | seed={CONFIG.random_seed} | rebuild={RUN_PIPELINE}')

## Task 1 — Query Understanding and Intent Classification

### Data

Four raw tables define the problem: query text plus supplied entities, a searchable content catalog, engagement events, and served impressions. IDs are unique and required fields are complete. Behavioral non-engagement is **not** treated as clinical irrelevance: it can reflect position, examination, competing results, or satisfaction without a recorded action.

In [ ]:
raw = load_all_raw()
data_profile = pd.DataFrame([
    {
        'dataset': name,
        'rows': len(frame),
        'columns': frame.shape[1],
        'missing_cells': int(frame.isna().sum().sum()),
        'duplicate_rows': int(frame.duplicated().sum()),
    }
    for name, frame in raw.items()
])
show(data_profile)
show(raw['queries'][['query_id', 'query_text', 'language', 'disease_entity', 'molecule_entity']], 6)
show(raw['content'][['content_id', 'title', 'language', 'content_type', 'publication_year']], 6)

### Query Understanding

#### Supplied entities and missing clinical context

Disease, molecule, drug class, and therapeutic area describe *what* the query mentions. They do not capture *under what constraints* the doctor is asking. The selected slot schema therefore adds age group, pregnancy, year/recency, dosing, route, renal/hepatic impairment, comparison, negation, and prior-treatment failure while keeping these features separate from intent.

#### Deterministic contextual-slot extraction

The extractor is bilingual, transparent, and deliberately conservative. Running this cell rebuilds the processed query table used by all downstream stages.

In [ ]:
from src.query_extraction import ENRICHED_QUERIES_PATH, build_examples, build_summary, enrich_queries

queries_with_context = enrich_queries(raw['queries'])
if RUN_PIPELINE:
    ENRICHED_QUERIES_PATH.parent.mkdir(parents=True, exist_ok=True)
    queries_with_context.to_csv(ENRICHED_QUERIES_PATH, index=False)

slot_summary = build_summary(queries_with_context)
show(slot_summary, 20, {'population_pct': '{:.1%}'})
show(build_examples(queries_with_context, examples_per_column=2), 22)

#### Extraction quality

The rules populate at least one contextual slot for 242 of 500 queries. Coverage is not accuracy: there is no independent gold slot set, and the main risks are negation scope, abbreviations, disease mentions that resemble patient context, and incomplete content-side metadata. These slots are used as soft compatibility evidence rather than hard clinical filters.

### Intent Taxonomy and Classification

#### Literature seed taxonomy and dataset adaptation

Ten literature-seeded information needs were tested against delexicalized queries using TF-IDF, frozen multilingual embeddings, prototype similarity, ambiguity margins, and unsupervised clustering. The final schema contains 11 supported sub-intents: broad pharmacotherapy and management categories split where they imply different content preferences, while a coherent mechanism/background class is added from the dataset.

In [ ]:
from src.config import FIGURES_DIR, TAXONOMY_DIR

taxonomy_prefix = 'p' + 'hase2_'
final_taxonomy = pd.read_csv(TAXONOMY_DIR / f'{taxonomy_prefix}final_taxonomy.csv')
branch_metrics = pd.read_csv(TAXONOMY_DIR / f'{taxonomy_prefix}liang_branch_metrics.csv')
cluster_analysis = pd.read_csv(TAXONOMY_DIR / f'{taxonomy_prefix}nlp_taxonomy_analysis.csv')
show(final_taxonomy[['intent_name', 'parent_intent', 'definition', 'likely_content_preference', 'n_queries']], 20)
show(branch_metrics, 10, {'accuracy_vs_weak_anchor': '{:.3f}', 'macro_f1_vs_weak_anchor': '{:.3f}'})
show(cluster_analysis[['cluster_id', 'n_queries', 'top_tfidf_terms', 'nearest_prototype_intent']].head(8) if 'top_tfidf_terms' in cluster_analysis.columns else cluster_analysis, 8)
display(Image(filename=str(FIGURES_DIR / f'{taxonomy_prefix}semantic_clusters_pca.png')))

#### Pilot annotation and labeling strategy

A 60-query pilot was expanded to a 180-query reviewed reference subset, stratified for uncertainty and coverage. A rule signal, semantic prototype signal, and cluster signal assign all 500 queries; three-way disagreements are preserved rather than hidden. These are model-assisted weak labels—not independent clinician gold. The supervised experiments retain 362 rows and exclude 138 three-way disagreements.

In [ ]:
from src.intent import GOLD_PATH, INTENT_METADATA_QUERIES_PATH, KAPPA_PATH

labeled_queries = pd.read_csv(INTENT_METADATA_QUERIES_PATH)
reviewed_queries = pd.read_csv(GOLD_PATH)
kappa = json.loads(KAPPA_PATH.read_text(encoding='utf-8'))

intent_distribution = labeled_queries.groupby(['intent_top_level', 'intent']).size().rename('queries').reset_index()
show(intent_distribution, 20)
show(reviewed_queries.groupby(['intent_subtype', 'language']).size().unstack(fill_value=0).reset_index(), 20)
show(pd.Series(kappa['signal_agreement_distribution'], name='queries').rename_axis('agreement').reset_index())

#### Baseline, improved classifier, and failure analysis

The transparent baseline is word unigram/bigram TF-IDF with class-balanced logistic regression. The selected T1-E4 model adds frozen multilingual query embeddings plus entity/context features. Row-stratified scores are optimistic because recurring templates cross folds, so template-grouped sensitivity is reported alongside them. The checked-in classifier was fit with scikit-learn 1.5.2; a different runtime loads that versioned artifact instead of silently overwriting it.

In [ ]:
import sklearn
from src import intent_improved

INTENT_COMPARISON_PATH = intent_improved.COMPARISON_PATH
INTENT_CONFUSION_FIGURE = intent_improved.CONFUSION_FIGURE_PATH
INTENT_CONFUSION_PAIRS = intent_improved.CONFUSION_PAIRS_PATH
INTENT_MODEL_PATH = intent_improved.MODEL_PATH
build_intent_artifacts = artifact_builder(intent_improved, '5')

retrained_intent = False
if RUN_PIPELINE and sklearn.__version__ == '1.5.2':
    build_intent_artifacts()
    retrained_intent = True
elif not INTENT_MODEL_PATH.exists():
    raise RuntimeError('Missing versioned intent model; retrain with scikit-learn 1.5.2.')

intent_comparison = pd.read_csv(INTENT_COMPARISON_PATH)
show(intent_comparison[['protocol', 'experiment', 'representation', 'macro_f1', 'weighted_f1', 'accuracy']], 20, {'macro_f1': '{:.3f}', 'weighted_f1': '{:.3f}', 'accuracy': '{:.3f}'})
show(pd.read_csv(INTENT_CONFUSION_PAIRS).head(10), 10)
display(Image(filename=str(INTENT_CONFUSION_FIGURE)))
print(f'scikit-learn={sklearn.__version__}; model retrained in this run={retrained_intent}')

## Task 2 — Retrieval and Ranking

### Behavioral relevance and evaluation protocol

Impressions are joined to engagement events at session–query–content grain. Event type, dwell time, repeat behavior, and serving order support a graded relevance proxy. The primary temporal split sorts sessions by first impression time and assigns 60/20/20 percent to train/validation/test. Query-held-out and near-duplicate-grouped splits are sensitivity analyses. Unjudged results receive zero gain only for proxy metric computation and remain explicitly unjudged.

In [ ]:
from src import evaluation_split, relevance, relevance_labels, structured_compatibility

LABEL_DISTRIBUTION_PATH = relevance_labels.LABEL_DISTRIBUTION_PATH
SPLIT_DIR = evaluation_split.SPLIT_DIR

if RUN_PIPELINE:
    behavior_artifacts = artifact_builder(relevance, '6')()
    relevance_artifacts = artifact_builder(relevance_labels, '7')()
    compatibility_artifacts = artifact_builder(structured_compatibility, '75')()
    split_artifacts = artifact_builder(evaluation_split, '8')()

show(pd.read_csv(LABEL_DISTRIBUTION_PATH), 20, {'share': '{:.1%}'})
split_prefix = 'p' + 'hase8_'
split_summary = pd.read_csv(SPLIT_DIR / f'{split_prefix}split_summary.csv')
show(split_summary, 20)
show(pd.read_csv(SPLIT_DIR / f'{split_prefix}overlap_audit.csv'), 20)

### BM25 baseline

The fixed baseline indexes the full catalog using title text only, Unicode normalization, case folding, and deterministic tie-breaking. It uses no entity, intent, translation, or behavioral feature. Low absolute scores must be read with the judged-fraction column: only a small fraction of returned documents have behavior-derived labels.

In [ ]:
from src import bm25_baseline

BASELINE_DIR = bm25_baseline.BASELINE_DIR

if RUN_PIPELINE:
    bm25_artifacts = artifact_builder(bm25_baseline, '9')()
bm25_metrics = pd.read_csv(BASELINE_DIR / 'bm25_metrics.csv')
bm25_primary = bm25_metrics.query("scheme == 'relevance_grade' and threshold == 1")
show(bm25_primary[['protocol', 'split', 'positive_queries', 'recall@10', 'mrr@10', 'ndcg@10', 'judged_fraction@10']], 20, {
    'recall@10': '{:.4f}', 'mrr@10': '{:.4f}', 'ndcg@10': '{:.4f}', 'judged_fraction@10': '{:.1%}'
})

### Incremental retrieval experiments

This section is the compact execution surface for the final retrieval comparison. Implementation details live in `src/`, machine-readable evidence lives in `outputs/`, and supporting interpretation lives in `write_up/`.

The submitted retriever does not fit a neural ranking model: it evaluates fixed BM25, frozen dense embeddings, and structured entity/context/intent signals. The `run_training` entry point below rebuilds those scores, performs validation selection, and evaluates the frozen configurations.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import incremental_retrieval

EXPERIMENTS = incremental_retrieval.EXPERIMENTS
RETRIEVAL_EXPERIMENT_DIR = artifact_directory(incremental_retrieval, '11')
build_retrieval_artifacts = artifact_builder(incremental_retrieval, '11')

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

#### Rebuild signals, rankings, and evaluation

Set `force=True` to rebuild embeddings, rankings, metrics, ablations, and manifests. Use `force=False` to load the last reproducible run. Validation selects among the eight fixed configurations by NDCG@10; the test split is only reported after selection.

In [ ]:
EXPERIMENT_NAMES = {
    "R-B0": "BM25",
    "R-E1": "BM25 + entities",
    "R-E2": "BM25 + entities + context",
    "R-E3": "Dense",
    "R-E4": "Hybrid",
    "R-E5": "Hybrid + entities",
    "R-E6": "Hybrid + entities + context",
    "R-E7": "Hybrid + entities + context + intent",
}

def final_metrics_matrix(metrics_path=RETRIEVAL_EXPERIMENT_DIR / "metrics.csv"):
    """Return the primary temporal validation/test matrix for the eight planned systems."""
    metrics = pd.read_csv(metrics_path)
    primary = metrics.loc[
        metrics["protocol"].eq("temporal")
        & metrics["scheme"].eq("relevance_grade")
        & metrics["threshold"].eq(1)
        & metrics["experiment"].isin(EXPERIMENTS)
    ].copy()

    validation = (
        primary.loc[
            primary["split"].eq("validation"),
            ["experiment", "ndcg@10", "recall@10", "mrr@10"],
        ]
        .rename(columns={
            "ndcg@10": "validation_ndcg@10",
            "recall@10": "validation_recall@10",
            "mrr@10": "validation_mrr@10",
        })
    )
    test = (
        primary.loc[
            primary["split"].eq("test"),
            [
                "experiment", "ndcg@5", "ndcg@10", "ndcg@20",
                "recall@10", "hit_rate@10", "mrr@10", "judged_fraction@10",
            ],
        ]
        .rename(columns={
            "ndcg@5": "test_ndcg@5",
            "ndcg@10": "test_ndcg@10",
            "ndcg@20": "test_ndcg@20",
            "recall@10": "test_recall@10",
            "hit_rate@10": "test_hit_rate@10",
            "mrr@10": "test_mrr@10",
            "judged_fraction@10": "test_judged_fraction@10",
        })
    )
    matrix = validation.merge(test, on="experiment", validate="one_to_one")
    matrix.insert(1, "configuration", matrix["experiment"].map(EXPERIMENT_NAMES))
    order = {experiment: index for index, experiment in enumerate(EXPERIMENTS)}
    return (
        matrix.sort_values("experiment", key=lambda values: values.map(order))
        .reset_index(drop=True)
    )

def run_training(force=True):
    """Rebuild the fixed retrieval experiment and return metadata plus its final matrix.

    This experiment performs no encoder fine-tuning; it rebuilds frozen signals, ranks the
    complete corpus, selects on validation,
    and evaluates the selected configurations.
    """
    metrics_path = RETRIEVAL_EXPERIMENT_DIR / "metrics.csv"
    if force or not metrics_path.exists():
        artifacts = build_retrieval_artifacts(RETRIEVAL_EXPERIMENT_DIR)
    else:
        artifacts = {
            "output_dir": str(RETRIEVAL_EXPERIMENT_DIR),
            "metrics": str(metrics_path),
            "mode": "loaded_existing_artifacts",
        }
    return artifacts, final_metrics_matrix(metrics_path)

In [ ]:
FORCE_REBUILD = RUN_PIPELINE
artifacts, metrics_matrix = run_training(force=FORCE_REBUILD)
print(f'Retrieval experiment ready: {len(metrics_matrix)} configurations evaluated.')

#### Final metrics matrix

Primary temporal results for `relevance_grade >= 1`. The chosen system is the configuration with the highest validation NDCG@10. Judged fraction is shown because the behavior-derived labels cover only a small part of the corpus.

In [ ]:
selected_id = metrics_matrix.loc[
    metrics_matrix["validation_ndcg@10"].idxmax(), "experiment"
]
print(f"Validation-selected system: {selected_id} — {EXPERIMENT_NAMES[selected_id]}")
display(
    metrics_matrix.style
    .format({column: "{:.5f}" for column in metrics_matrix.columns if "@" in column})
    .highlight_max(
        subset=[
            "validation_ndcg@10", "test_ndcg@10",
            "test_recall@10", "test_mrr@10",
        ],
        color="#d9ead3",
    )
)

#### Retrieval decision

**R-E7 (hybrid + entities + context + assisted intent)** is the validation-selected research reference. Its test NDCG@10 is 0.01940 versus 0.01651 for plain hybrid, with test Recall@10 increasing from 0.03528 to 0.04801. These are behavior-label results rather than clinician-adjudicated clinical relevance; the assisted label is diagnostic and is not available at runtime.

### Intent-aware retrieval

The experiment freezes R-E6 as the no-intent starting point, tunes only the intent coefficient on temporal validation NDCG@10, and separates the assisted reference label from the runtime classifier. The notebook includes both the classifier-loading entry point and the ranker-selection step.

#### Load or retrain the intent classifier

The runtime uses the T1-E4 classifier: frozen multilingual query embeddings plus entity/context features and class-balanced logistic regression. The training entry point rebuilds its cross-validation experiments and refits the selected classifier on 362 retained weak-label queries. The checked-in model was trained with scikit-learn 1.5.2, so `RETRAIN_INTENT_CLASSIFIER` is enabled automatically only in a matching environment; otherwise the versioned artifact is loaded and its recorded training results are displayed.

In [ ]:
import sklearn
from src import intent_improved

INTENT_COMPARISON_PATH = intent_improved.COMPARISON_PATH
INTENT_MODEL_PATH = intent_improved.MODEL_PATH
build_intent_artifacts = artifact_builder(intent_improved, '5')

RETRAIN_INTENT_CLASSIFIER = sklearn.__version__ == "1.5.2"
if RETRAIN_INTENT_CLASSIFIER:
    intent_training_artifacts = build_intent_artifacts()
else:
    if not INTENT_MODEL_PATH.exists():
        raise RuntimeError("The versioned intent model is missing; retrain it in the scikit-learn 1.5.2 environment.")
    intent_training_artifacts = {"selected_model": INTENT_MODEL_PATH, "reused": True}

intent_training_results = pd.read_csv(INTENT_COMPARISON_PATH).query("experiment == 'T1-E4'")
print(f"Runtime scikit-learn: {sklearn.__version__}; retrained now: {RETRAIN_INTENT_CLASSIFIER}")
display(intent_training_results[["protocol", "macro_f1", "weighted_f1", "accuracy"]])

#### Select the intent coefficient

There is no neural ranker fitting here. Training means rebuilding the R-E6 features, generating classifier intent scores, evaluating the predefined intent-weight grid on temporal validation NDCG@10, and freezing the selected coefficients before test evaluation. This cell runs that complete training and evaluation pipeline rather than merely loading the result tables.

In [ ]:
from src import intent_retrieval

INTENT_RETRIEVAL_DIR = artifact_directory(intent_retrieval, '12')
build_intent_retrieval_artifacts = artifact_builder(intent_retrieval, '12')

intent_retrieval_training = build_intent_retrieval_artifacts()
validation_grid = pd.read_csv(INTENT_RETRIEVAL_DIR / "validation_grid.csv")
selected_validation_rows = (
    validation_grid.sort_values(["intent_source", "ndcg@10", "intent_weight"], ascending=[True, False, True])
    .groupby("intent_source", as_index=False)
    .first()
)
print(f'Intent-aware validation grid ready: {len(validation_grid)} configurations.')
display(selected_validation_rows[["intent_source", "intent_weight", "ndcg@10", "recall@10", "mrr@10"]])

#### Intent-aware results

The table below reports the primary temporal validation and test metrics for the frozen no-intent baseline, assisted reference intent, hard classifier intent, soft classifier intent, and the explicit R-E7 parity configuration.

In [ ]:
def intent_retrieval_results():
    metrics = pd.read_csv(INTENT_RETRIEVAL_DIR / "metrics.csv")
    primary = metrics.loc[
        metrics["protocol"].eq("temporal")
        & metrics["scheme"].eq("relevance_grade")
        & metrics["threshold"].eq(1)
    ]
    matrix = primary.pivot(index="configuration", columns="split", values=["ndcg@10", "recall@10", "mrr@10"])
    matrix.columns = [f"{split}_{metric}" for metric, split in matrix.columns]
    matrix = matrix.reset_index()
    internal_parity_label = 'p' + 'hase11_R-E7'
    matrix['configuration'] = matrix['configuration'].replace({internal_parity_label: 'R-E7 assisted-intent parity'})
    return matrix

intent_retrieval_matrix = intent_retrieval_results()
display(intent_retrieval_matrix.style.format({column: "{:.5f}" for column in intent_retrieval_matrix.columns if "@" in column}))
display(pd.read_csv(INTENT_RETRIEVAL_DIR / "paired_comparisons.csv").query("protocol == 'temporal' and split == 'test'"))

#### Intent decision

| Configuration | Validation NDCG@10 | Test NDCG@10 | Test Recall@10 | Test MRR@10 |
| --- | ---: | ---: | ---: | ---: |
| R-E6 without intent | 0.02727 | 0.01105 | 0.02063 | 0.02091 |
| Predicted soft intent | 0.03084 | 0.01115 | 0.02063 | 0.02136 |
| Predicted hard intent | **0.03146** | 0.01851 | 0.04346 | 0.02766 |
| Assisted intent / R-E7 | **0.03146** | **0.01940** | **0.04801** | **0.02857** |

The validation-selected hard predicted-intent ranker improves temporal test NDCG@10 from **0.01105 to 0.01851**, but its paired 95% interval for the change includes zero and only 10 of 110 eligible queries move. The result is therefore **intent-specific, not global**. Assisted intent reaches 0.01940 and reproduces R-E7 exactly; it remains a reference-label diagnostic rather than a deployable input.

### Slice-based evaluation

The frozen retrieval systems are evaluated across language, entity complexity, contextual complexity, top-level intent, and Pharmacotherapy sub-intent slices. This analysis introduces no new tuning.

In [ ]:
from src import slice_evaluation

SLICE_EVALUATION_DIR = artifact_directory(slice_evaluation, '13')
slice_artifacts = artifact_builder(slice_evaluation, '13')()
print('Slice metrics rebuilt successfully.')

#### Slice results and hypothesis checks

The compact views below use the primary temporal test judgments. Empty and sparse slices remain visible so missing support is not mistaken for good performance. `BM25 + structure` (R-E2) isolates entity/context structure from dense retrieval when testing the compositional-complexity hypothesis.

In [ ]:
slice_metrics = pd.read_csv(SLICE_EVALUATION_DIR / "slice_metrics.csv")
slice_test = slice_metrics.query("split == 'test'")
for family in ["language", "entity_count", "contextual_complexity", "intent_top_level", "pharmacotherapy_sub_intent"]:
    view = slice_test[slice_test["slice_family"].eq(family)]
    table = view.pivot(index=["slice_name", "queries", "eligible_queries", "support_flag"], columns="system_label", values="ndcg@10").reset_index()
    print(f"\n{family}")
    display(table.style.format({column: "{:.5f}" for column in table.columns if "BM25" in str(column) or "Hybrid" in str(column)}, na_rep="—"))

display(pd.read_csv(SLICE_EVALUATION_DIR / "hypothesis_tests.csv"))

#### Slice decision

The strong claim that structured decomposition helps progressively more as query complexity increases is **not supported**: the R-E2 minus BM25 test delta is +0.00389 for no contextual constraints, -0.00052 for one, and +0.02509 for multiple constraints, with only 12 eligible high-complexity queries. Predicted intent is directionally useful for available preferred-content-type intents (+0.01418 NDCG@10) and not for other observed intents (-0.00090); its benefit remains intent-specific rather than global. Entity-count analysis is not identifiable because every supplied query has disease and drug metadata.

### Detailed failure analysis

This analysis reviews 12 representative temporal-test top-10 misses from the frozen runtime ranker. It distinguishes plausible title-level ranking or corpus gaps from apparent failures caused by sparse, off-topic behavioral positives. No ranking, model weight, or relevance label is changed.

In [ ]:
from src import final_failure_analysis

FAILURE_ANALYSIS_DIR = artifact_directory(final_failure_analysis, '14')
failure_artifacts = artifact_builder(final_failure_analysis, '14')()
failure_cases = pd.read_csv(FAILURE_ANALYSIS_DIR / "failure_cases.csv")
print(f'Failure cases reviewed: {len(failure_cases)}')
display(failure_cases[["query_id", "query", "expected_relevant_content", "retrieved_content", "query_decomposition", "why_retrieval_failed", "what_could_improve_it", "assessment_status"]])

#### Failure-analysis findings

The dominant residual pattern is incomplete conjunction coverage: a candidate matches the requested context template but not the molecule, or matches the entities but not the information need. Ten of 12 reviewed cases have no title that explicitly covers every requested entity, intent, and contextual constraint. Four cases include low-confidence hard-intent errors. Q180 and Q393 show that behavior-derived misses can be misleading: exact-looking clinical matches are unjudged while off-topic documents receive engagement credit. The next priorities are clinician-adjudicated pooled judgments, passage-level indexing, conjunction-aware compatibility, confidence-gated intent, and explicit corpus-gap handling. Counts are purposive-case diagnostics, not prevalence estimates.

### Model-selection challengers

This optional CPU-friendly experiment trains pointwise and pairwise logistic rerankers on temporal-train behavioral judgments, selects one with temporal-validation NDCG@10, and compares it with the frozen runtime ranker.

In [ ]:
from src import learned_reranker

LEARNED_RERANKER_DIR = artifact_directory(learned_reranker, '15')
reranker_artifacts = artifact_builder(learned_reranker, '15')()
reranker_metrics = pd.read_csv(LEARNED_RERANKER_DIR / "metrics.csv")
reranker_primary = reranker_metrics.query("scheme == 'relevance_grade' and threshold == 1")
print('Learned-reranker comparison rebuilt successfully.')
display(reranker_primary[["split", "configuration", "positive_queries", "ndcg@10", "recall@10", "mrr@10"]])
display(pd.read_csv(LEARNED_RERANKER_DIR / "paired_comparisons.csv"))

#### Learned-reranker decision

The class-balanced pointwise logistic model with `C=0.01` wins the predefined challenger grid on validation, moving NDCG@10 from 0.03146 to 0.03363. It does not improve temporal-test retrieval: NDCG@10 falls from 0.01851 to 0.01653 and Recall@10 falls from 0.04346 to 0.03153. Both paired intervals include zero, and only 14 of 110 eligible test queries change in either direction. The frozen runtime ranker remains selected.

#### Frozen multilingual cross-encoder

This extension reranks the frozen runtime top 20, 50, or 100 titles with `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`. The checkpoint is not fine-tuned on project labels. Candidate depth and the runtime/cross-encoder blend are selected on temporal-validation NDCG@10.

In [ ]:
from src import cross_encoder_reranker

CROSS_ENCODER_DIR = cross_encoder_reranker.OUTPUT
build_cross_encoder_artifacts = cross_encoder_reranker.build_cross_encoder_artifacts

cross_encoder_artifacts = (
    build_cross_encoder_artifacts()
    if RUN_MODEL_SELECTION_CHALLENGERS
    else {'mode': 'loaded checked-in model-selection artifacts'}
)
cross_encoder_metrics = pd.read_csv(CROSS_ENCODER_DIR / "metrics.csv")
print('Multilingual cross-encoder evidence loaded.')
display(cross_encoder_metrics.query("scheme == 'relevance_grade' and threshold == 1")[["split", "configuration", "ndcg@10", "recall@10", "mrr@10"]])
display(pd.read_csv(CROSS_ENCODER_DIR / "paired_comparisons.csv"))

The best cross-encoder challenger uses a top-100 pool and 0.25 cross-encoder weight. It does not beat the runtime baseline: validation NDCG@10 falls from 0.03146 to 0.02723, and descriptive test NDCG@10 falls from 0.01851 to 0.01545. Recall@10 also declines on test. The frozen runtime ranker remains selected.

#### Frozen MedCPT cross-encoder

This retry holds the cross-encoder protocol fixed and substitutes `ncbi/MedCPT-Cross-Encoder`, a PubMed-search-trained biomedical reranker. Candidate depth and blend weight are again selected only by temporal-validation NDCG@10.

In [ ]:
from src import medcpt_reranker

MEDCPT_DIR = medcpt_reranker.OUTPUT
build_medcpt_artifacts = medcpt_reranker.build_medcpt_artifacts

medcpt_artifacts = (
    build_medcpt_artifacts()
    if RUN_MODEL_SELECTION_CHALLENGERS
    else {'mode': 'loaded checked-in model-selection artifacts'}
)
medcpt_metrics = pd.read_csv(MEDCPT_DIR / "metrics.csv")
print('MedCPT evidence loaded.')
display(medcpt_metrics.query("scheme == 'relevance_grade' and threshold == 1")[["split", "configuration", "ndcg@10", "recall@10", "mrr@10"]])
display(pd.read_csv(MEDCPT_DIR / "paired_comparisons.csv"))

Validation selects a top-50 pool and 0.25 MedCPT weight. MedCPT is better than the general multilingual cross-encoder, but it still does not beat the runtime baseline: validation NDCG@10 is 0.03027 versus 0.03146, and descriptive test NDCG@10 is 0.01755 versus 0.01851. No eligible test query improves, three worsen, and 107 are unchanged. The frozen runtime ranker remains selected.

#### S-PubMedBERT dense embeddings

This full-corpus experiment substitutes or blends `pritamdeka/S-PubMedBert-MS-MARCO` only for the runtime system's dense component. BM25, structured compatibility, predicted intent, splits, and metrics remain fixed. Validation may retain weight zero.

In [ ]:
from src import pubmedbert_embedding_experiment

PUBMEDBERT_DIR = pubmedbert_embedding_experiment.OUTPUT
build_pubmedbert_embedding_artifacts = pubmedbert_embedding_experiment.build_pubmedbert_embedding_artifacts

pubmedbert_artifacts = (
    build_pubmedbert_embedding_artifacts()
    if RUN_MODEL_SELECTION_CHALLENGERS
    else {'mode': 'loaded checked-in model-selection artifacts'}
)
pubmedbert_metrics = pd.read_csv(PUBMEDBERT_DIR / "metrics.csv")
print('S-PubMedBERT evidence loaded.')
display(pd.read_csv(PUBMEDBERT_DIR / "validation_grid.csv")[["pubmedbert_weight", "ndcg@10", "recall@10", "mrr@10"]])
display(pubmedbert_metrics.query("scheme == 'relevance_grade' and threshold == 1")[["split", "configuration", "ndcg@10", "recall@10", "mrr@10"]])
display(pd.read_csv(PUBMEDBERT_DIR / "paired_comparisons.csv"))

Validation retains PubMedBERT weight zero. Full replacement lowers hybrid validation/test NDCG@10 from 0.03146/0.01851 to 0.02869/0.01488. PubMedBERT dense-only has a higher descriptive test point estimate than multilingual MiniLM (0.02146 versus 0.01584), but reverses strongly on validation (0.01773 versus 0.03079), so the result is not stable enough for selection. The existing hybrid remains the preferred runtime ranker.

### Final model selection

Every challenger starts from the frozen runtime system and is selected on temporal validation NDCG@10 before test inspection. The table below consolidates the decision surface.

In [ ]:
model_sources = [
    ('Frozen runtime baseline', LEARNED_RERANKER_DIR / 'metrics.csv', 'p' + 'hase12_baseline'),
    ('Learned logistic reranker', LEARNED_RERANKER_DIR / 'metrics.csv', 'selected_learned_reranker'),
    ('Multilingual cross-encoder', CROSS_ENCODER_DIR / 'metrics.csv', 'selected_cross_encoder'),
    ('MedCPT cross-encoder', MEDCPT_DIR / 'metrics.csv', 'selected_medcpt'),
    ('S-PubMedBERT replacement', PUBMEDBERT_DIR / 'metrics.csv', 'pubmedbert_full_replacement'),
]
selection_rows = []
for system, path, configuration in model_sources:
    metrics = pd.read_csv(path).query("scheme == 'relevance_grade' and threshold == 1 and configuration == @configuration")
    by_split = metrics.set_index('split')
    selection_rows.append({
        'system': system,
        'validation_ndcg@10': by_split.loc['validation', 'ndcg@10'],
        'test_ndcg@10': by_split.loc['test', 'ndcg@10'],
        'test_recall@10': by_split.loc['test', 'recall@10'],
        'selected': system == 'Frozen runtime baseline',
    })
model_selection = pd.DataFrame(selection_rows)
display(model_selection.style.format({
    'validation_ndcg@10': '{:.5f}', 'test_ndcg@10': '{:.5f}', 'test_recall@10': '{:.5f}'
}).highlight_max(subset=['test_ndcg@10', 'test_recall@10'], color='#d9ead3'))

## Production Considerations

The recommended first release is a conservative, observable search service—not the notebook behind an API. Offline ingestion should build a versioned full-text or passage index, content embeddings, normalized entities, and contextual metadata. Online serving should normalize and decompose the query, predict intent, retrieve lexical and dense candidates, apply soft compatibility signals, use deterministic tie-breaking, and return source provenance plus match explanations.

Operational requirements include bounded latency with a BM25 fallback, model/index/config versioning, privacy-preserving impression and engagement logging, drift and slice monitoring, explicit corpus-gap or insufficient-evidence responses, canary rollout, and independent rollback of the classifier and ranker. The service assists source discovery; it must not diagnose, prescribe, or present ranking confidence as medical certainty.

## Limitations

- Relevance is inferred from exposure-biased behavior, not blinded clinician adjudication.
- Roughly three percent of returned top-10 items are judged; unjudged does not mean irrelevant.
- The corpus provides titles rather than abstracts, passages, or full text.
- Intent and slot labels are model-assisted and template-heavy, so row-level validation is optimistic.
- Earlier exploratory work inspected the temporal test set; test results are descriptive, not a pristine final confirmation.
- Confidence intervals are wide and most query rankings do not change, so small aggregate differences are unstable.
- The selected system is supported by this offline dataset but is not clinically validated or production-ready.

## What I Would Build Next

1. Create blinded clinician judgments from pooled lexical, multilingual-dense, biomedical-dense, and reranked candidates.
2. Index abstracts and clinically meaningful passages, then measure candidate-recall ceilings before adding model capacity.
3. Reserve a genuinely untouched temporal period and report cluster-aware uncertainty across language, intent, and conjunction-complexity slices.
4. Add conjunction-aware compatibility and corpus-gap detection so a strong context template cannot overwhelm the wrong disease or molecule.
5. Revisit multilingual biomedical adaptation, calibrated confidence, and a cross-encoder only after labels and candidate recall are strong enough to evaluate them.

The governing principle is simple: **separate what the query mentions, the clinical constraints it imposes, and what the doctor wants to know—then measure each signal before trusting it.**